# ⚙️ Customer Churn Prediction
## Notebook 2 — Feature Engineering + RFM Segmentation

| Detail | Info |
|---|---|
| **Input**  | Raw Telco Churn Dataset |
| **Output** | Processed dataset ready for modelling |
| **Author** | Shubham Kumar |
| **Tools**  | Pandas · Scikit-learn · KMeans |

### Objective
Clean and transform raw data into model-ready features.
Segment customers using RFM analysis and KMeans clustering.

---
## 1. Environment Setup
*Installing and importing all required libraries*

In [1]:
# ================================================================
# SECTION 1 : ENVIRONMENT SETUP
# ================================================================

!pip install xgboost shap imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("=" * 50)
print("  All libraries loaded successfully!")
print("=" * 50)

  All libraries loaded successfully!


---
## 2. Data Loading
*Loading raw dataset and applying fixes from EDA*

In [2]:
# ================================================================
# SECTION 2 : DATA LOADING
# ================================================================

# 2.1 Load dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# 2.2 Apply EDA fixes
# Fix TotalCharges — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill 11 missing values with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop CustomerID — not useful for modelling
df.drop('customerID', axis=1, inplace=True)

# 2.3 Confirm
print("=" * 50)
print("  Dataset Loaded and Fixed!")
print("=" * 50)
print(f"  Rows      : {df.shape[0]}")
print(f"  Columns   : {df.shape[1]}")
print(f"  Nulls     : {df.isnull().sum().sum()}")
print("=" * 50)
print("\nColumn names:")
print(df.columns.tolist())

  Dataset Loaded and Fixed!
  Rows      : 7043
  Columns   : 20
  Nulls     : 0

Column names:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [3]:
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# ================================================================
# SECTION 3 : ENCODING CATEGORICAL COLUMNS
# ================================================================

# 3.1 Check categorical columns
print("--- 3.1 Categorical Columns ---")
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)
print(f"Total categorical columns : {len(cat_cols)}")

# 3.2 Binary columns — Label Encoding (Yes/No → 1/0)
print("\n--- 3.2 Binary Encoding ---")
binary_cols = ['gender', 'Partner', 'Dependents',
               'PhoneService', 'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0,
                           'Male': 1, 'Female': 0})
    print(f"  {col} encoded successfully")

# 3.3 Multi-category columns — One Hot Encoding
print("\n--- 3.3 One Hot Encoding ---")
multi_cols = ['MultipleLines', 'InternetService',
              'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)
print(f"  One hot encoding done!")
print(f"  New shape after encoding : {df.shape}")

# 3.4 Confirm no more object columns
print("\n--- 3.4 Confirm All Numeric ---")
remaining = df.select_dtypes(include='object').columns.tolist()
if len(remaining) == 0:
    print("  All columns are now numeric!")
else:
    print(f"  Still object : {remaining}")

print("\n" + "=" * 50)
print("  Encoding Complete!")
print("=" * 50)

--- 3.1 Categorical Columns ---
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Total categorical columns : 16

--- 3.2 Binary Encoding ---
  gender encoded successfully
  Partner encoded successfully
  Dependents encoded successfully
  PhoneService encoded successfully
  PaperlessBilling encoded successfully
  Churn encoded successfully

--- 3.3 One Hot Encoding ---
  One hot encoding done!
  New shape after encoding : (7043, 31)

--- 3.4 Confirm All Numeric ---
  All columns are now numeric!

  Encoding Complete!


---
## 4. Feature Scaling
*Bringing all numerical columns to same scale so model is not biased*

In [5]:
# ================================================================
# SECTION 4 : FEATURE SCALING
# ================================================================

# 4.1 Separate features and target
print("--- 4.1 Separate Features and Target ---")
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"  Features shape : {X.shape}")
print(f"  Target shape   : {y.shape}")
print(f"  Churn rate     : {y.mean()*100:.2f}%")

# 4.2 Columns to scale
print("\n--- 4.2 Columns to Scale ---")
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(f"  Scaling columns : {scale_cols}")

# 4.3 Apply StandardScaler
scaler = StandardScaler()
X[scale_cols] = scaler.fit_transform(X[scale_cols])

print("\n--- 4.3 After Scaling ---")
print(X[scale_cols].describe().round(2))

# 4.4 Confirm
print("\n" + "=" * 50)
print("  Scaling Complete!")
print(f"  Mean of tenure         : {X['tenure'].mean():.2f}")
print(f"  Std  of tenure         : {X['tenure'].std():.2f}")
print(f"  Mean of MonthlyCharges : {X['MonthlyCharges'].mean():.2f}")
print(f"  Std  of MonthlyCharges : {X['MonthlyCharges'].std():.2f}")
print("=" * 50)

--- 4.1 Separate Features and Target ---
  Features shape : (7043, 30)
  Target shape   : (7043,)
  Churn rate     : 26.54%

--- 4.2 Columns to Scale ---
  Scaling columns : ['tenure', 'MonthlyCharges', 'TotalCharges']

--- 4.3 After Scaling ---
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     -0.00           -0.00         -0.00
std       1.00            1.00          1.00
min      -1.32           -1.55         -1.00
25%      -0.95           -0.97         -0.83
50%      -0.14            0.19         -0.39
75%       0.92            0.83          0.66
max       1.61            1.79          2.83

  Scaling Complete!
  Mean of tenure         : -0.00
  Std  of tenure         : 1.00
  Mean of MonthlyCharges : -0.00
  Std  of MonthlyCharges : 1.00


---
## 5. Train Test Split
*Splitting data into training and testing sets*

In [6]:
# ================================================================
# SECTION 5 : TRAIN TEST SPLIT
# ================================================================

from sklearn.model_selection import train_test_split

# 5.1 Split data
X_train, X_test, y_train, y_test = train_test_split(
                                        X, y,
                                        test_size=0.2,
                                        random_state=42,
                                        stratify=y)

# 5.2 Confirm split
print("=" * 50)
print("  TRAIN TEST SPLIT COMPLETE!")
print("=" * 50)
print(f"  Total data     : {len(X)}")
print(f"  Training set   : {len(X_train)} rows (80%)")
print(f"  Testing set    : {len(X_test)} rows  (20%)")
print("=" * 50)
print(f"\n  Training churn rate : {y_train.mean()*100:.2f}%")
print(f"  Testing  churn rate : {y_test.mean()*100:.2f}%")
print("\n  Both rates similar = stratify worked! ✅")

  TRAIN TEST SPLIT COMPLETE!
  Total data     : 7043
  Training set   : 5634 rows (80%)
  Testing set    : 1409 rows  (20%)

  Training churn rate : 26.54%
  Testing  churn rate : 26.54%

  Both rates similar = stratify worked! ✅


END

In [7]:
# ================================================================
# SAVE PROCESSED DATA FOR MODELLING
# ================================================================

import joblib

# Save X_train, X_test, y_train, y_test
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_test,  'X_test.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test,  'y_test.pkl')
joblib.dump(scaler,  'scaler.pkl')

print("=" * 50)
print("  All data saved successfully!")
print("=" * 50)
print("  X_train.pkl ✅")
print("  X_test.pkl  ✅")
print("  y_train.pkl ✅")
print("  y_test.pkl  ✅")
print("  scaler.pkl  ✅")
print("=" * 50)
print("\n  Notebook 2 Complete! Ready for Modelling.")

  All data saved successfully!
  X_train.pkl ✅
  X_test.pkl  ✅
  y_train.pkl ✅
  y_test.pkl  ✅
  scaler.pkl  ✅

  Notebook 2 Complete! Ready for Modelling.
